# 建立環境


In [ ]:
!pip install pandas

# 開始進行OCR，並且彙整進excel中

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
import os
from PIL import Image

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Install the Tesseract OCR engine (system level)
!apt install tesseract-ocr -y

# Install the pytesseract Python wrapper
!pip install pytesseract

In [ ]:
import pytesseract

In [ ]:
!apt update
!apt install -y tesseract-ocr-chi-tra

In [ ]:
# Make sure all required libraries are imported
import pandas as pd
import os
from PIL import Image
from natsort import natsorted # Import natsorted for natural sorting

# !apt update
# !apt install tesseract-ocr -y
# !apt install -y tesseract-ocr-chi-tra
# Install natsort if not already installed
!pip install natsort


# --- Step 1: copy the dataset folder from Google Drive into Colab ---
# Path to the dataset folder in Google Drive
drive_folder_path = '/content/drive/MyDrive/大三/下/AI導論/詐騙正常對話資料集'
# Target path inside Colab
colab_folder_path = '/content/詐騙正常對話資料集_colab'

# Create the target folder in Colab if it does not exist
os.makedirs(colab_folder_path, exist_ok=True)

# Copy the files
# rsync is faster when there are many files
print(f"Copying from '{drive_folder_path}' to '{colab_folder_path}'...")
!rsync -av "{drive_folder_path}/" "{colab_folder_path}/"
print("Folder copy complete.")

# --- Step 2: run OCR to extract Chinese text from the images ---
# Collect paths to every image in the folder
image_files = [os.path.join(colab_folder_path, f) for f in os.listdir(colab_folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
image_files = natsorted(image_files) # natural sort

print(f"\n'{colab_folder_path}' contains {len(image_files)} images.")
if not image_files:
    print("Error: no image files found. Check the folder path and file extensions.")
    print("Supported formats are .png, .jpg, .jpeg, .gif and .bmp.")
    print("Also confirm the dataset folder exists in your Google Drive.")

# List holding the OCR results
# Each entry is a dict with the filename and the extracted text
ocr_results_list = []

print("\nStarting OCR extraction. This may take a while...")
for i, image_path in enumerate(image_files):
    filename = os.path.basename(image_path)
    try:
        img = Image.open(image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_results_list.append({'檔名': filename, '辨識內容': text.strip()})
        print(f"Processed image {i+1}/{len(image_files)}: {filename}")
        if text.strip():
            print(f"  OCR succeeded, sample text: \"{text.strip()[:50]}...\"") # show the first 50 chars
        else:
            print(f"  OCR ran, but no text was detected. The image text may be unclear, or Tesseract could not read it.")
    except Exception as e:
        ocr_results_list.append({'檔名': filename, '辨識內容': f"處理圖片失敗: {e}"})
        print(f"Processing image {i+1}/{len(image_files)} failed: {filename} - error: {e}")

print("\nOCR extraction complete.")

# Preview the OCR results
if ocr_results_list:
    ocr_df = pd.DataFrame(ocr_results_list)
    print("\n--- OCR results preview (first 5 rows) ---")
    print(ocr_df.head())
    print(f"Extracted text from {len(ocr_df)} images.")
else:
    print("\nNo OCR results. Check whether any images were processed.")

In [ ]:
import pandas as pd
import os

# Assumes the OCR cell above has run and `ocr_results_list` is populated.
# To make this cell self-contained after a restart, images are reloaded
# and OCR is re-run.
# If you are running cells in the same session, ocr_results_list is already available.

# --- Re-check folder paths and image files (if running in a fresh Colab session) ---
colab_folder_path = '/content/詐騙正常對話資料集_colab'

# If the Colab session was restarted, re-mount Drive and copy the files again:
# from google.colab import drive
# drive.mount('/content/drive')
# drive_folder_path = '/content/drive/MyDrive/詐騙正常對話資料集'
# !rsync -av "{drive_folder_path}/" "{colab_folder_path}/"

# Collect paths to every image in the folder
image_files = [os.path.join(colab_folder_path, f) for f in os.listdir(colab_folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
from natsort import natsorted
image_files = natsorted(image_files) # natural sort for consistent ordering

# If ocr_results_list is gone (e.g. after a restart), re-run OCR
# otherwise it is still in memory
if 'ocr_results_list' not in locals():
    import pytesseract
    from PIL import Image
    ocr_results_list = []
    print("'ocr_results_list' is not in memory; re-running OCR...")
    for i, image_path in enumerate(image_files):
        filename = os.path.basename(image_path)
        try:
            img = Image.open(image_path)
            text = pytesseract.image_to_string(img, lang='chi_tra')
            ocr_results_list.append({'檔名': filename, '辨識內容': text.strip()})
        except Exception as e:
            ocr_results_list.append({'檔名': filename, '辨識內容': f"處理圖片失敗: {e}"})
    print("Re-ran OCR extraction.")

# --- Create the DataFrame ---
# Convert the OCR results into a pandas DataFrame
df = pd.DataFrame(ocr_results_list)

# Rename columns for the Excel export
df = df.rename(columns={'辨識內容': '對話內容'})

# Add an empty conversation-type column for manual labelling
df['對話類型'] = ''

# Pre-fill the label from the filename where possible
# Labels are inferred from keywords in the filename
# Adjust to match your own naming convention
for i, row in df.iterrows():
    filename = row['檔名']
    if "詐騙" in filename:
        df.loc[i, '對話類型'] = '詐騙'
    elif "正常" in filename:
        df.loc[i, '對話類型'] = '正常'
    else:
        df.loc[i, '對話類型'] = '待標記' # mark as pending if the filename is ambiguous


# Check that the DataFrame has rows
print("\n--- DataFrame preview (first 5 rows) ---")
print(df.head())
print(f"DataFrame contains {len(df)} rows.")

# --- Export to Excel ---
excel_output_path = '/content/對話辨識結果.xlsx'

try:
    # Write the selected columns to Excel in order
    # conversation text goes in column A, label in column B
    df[['對話內容', '對話類型']].to_excel(excel_output_path, index=False, sheet_name='對話紀錄', startrow=0)
    print(f"\nOCR results exported to Excel: '{excel_output_path}'")
    print("Download the Excel file and label each row in the conversation-type column as scam or normal.")
except Exception as e:
    print(f"Failed to write the Excel file: {e}")

# 讀取OCR過後的csv檔


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

In [ ]:
import pandas as pd
from google.colab import files

# --- Upload the CSV file ---
print("Upload your CSV file.")
uploaded = files.upload() # This will open a file selection dialog

# Get the name of the uploaded file
csv_file_name = next(iter(uploaded))
print(f"File '{csv_file_name}'  uploaded successfully.")

# --- Read the CSV file into a Pandas DataFrame with specified encoding ---
try:
    # First, try 'Big5' for Traditional Chinese
    df = pd.read_csv(csv_file_name, encoding='Big5')
    print("\nCSV loaded into a DataFrame (Big5 encoding).")

    # --- Display the DataFrame as a table ---
    print("\nHere is your data:")
    from IPython.display import display
    display(df)

except UnicodeDecodeError:
    # If Big5 fails, try 'GBK' (another common encoding for Chinese)
    try:
        df = pd.read_csv(csv_file_name, encoding='GBK')
        print("\nCSV loaded into a DataFrame (GBK encoding).")
        # --- Display the DataFrame as a table ---
        print("\nHere is your data:")
        from IPython.display import display
        display(df)
    except Exception as e_gbk:
        print(f"Error reading the CSV file: {e_gbk}")
        print("Both Big5 and GBK failed. Check that the file is valid CSV, or save it as UTF-8.")
except Exception as e:
    print(f"Error reading the CSV file: {e}")
    print("Check that the file is valid CSV.")

# 開始進行openai及finetuning


In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI

In [ ]:
import os
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model = "gpt-3.5-turbo",
    messages = [{"role":"system", "content":"你是一個人工智慧詐騙與正常訊息辨識助手，你可以幫我辨識我上傳的照片是詐騙對話還是正常對話。請你不要加入其他訊息與資訊，只要給我最原始的資料即可，不用做過多解釋."},
                {"role": "user", "content":"我上傳的對話內容截圖在這里:"},
                {"role": "assistant", "content": "告訴我是詐騙對話還是正常對話"}]
    )


In [ ]:
response.choices[0].message.content

In [ ]:
result = []

for i in range(107):

  response = client.chat.completions.create(
    model = "gpt-3.5-turbo",
    messages = [{"role":"system", "content":"你是一個人工智慧詐騙與正常訊息辨識助手，你可以幫我辨識我上傳的照片是詐騙對話還是正常對話。請你不要加入其他訊息與資訊，只要給我最原始的資料即可，不用做過多解釋."},
                {"role": "user", "content":"我上傳的對話內容截圖在這里:"},
                {"role": "assistant", "content": "告訴我是詐騙對話還是正常對話"}]
    )

  print(response.choices[0].message.content)
  result.append(response.choices[0].message.content)

# gpt-3.5-turbo的準確率(104初始資料)

In [ ]:
import time # for pacing the output
import os   # file path handling
from google.colab import drive # for mounting Google Drive
from PIL import Image # image handling
import pytesseract # OCR
from openai import OpenAI # OpenAI API
import re # regular expressions
from natsort import natsorted # install if missing: !pip install natsort
import pandas as pd # pandas, for displaying results

# --- 1. Configure the API key and fine-tuned model ID ---
# Replace with your own valid API key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # use your own API key

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
# the ! prefix runs a shell command in Colab
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
!pip install natsort > /dev/null # make sure natsort is installed
print("OCR environment ready.")

# --- 3. Mount Google Drive and define folder paths ---
print("\nMounting Google Drive...")
drive.mount('/content/drive')
print("Google Drive mounted.")

# Path to your dataset folder in Google Drive
drive_folder_path = '/content/drive/MyDrive/大三/下/AI導論/詐騙正常對話資料集' # adjust to your own folder

# Helper: Reduce the model output to a simple label
def get_simplified_label(text):
    if isinstance(text, str): # make sure the input is a string
        text_lower = text.lower()
        if "正常" in text_lower:
            return "正常"
        elif "詐騙" in text_lower:
            return "詐騙"
    return "未知"

# Check the folder exists
if not os.path.exists(drive_folder_path):
    print(f"Error: the folder '{drive_folder_path}' does not exist. Check the path.")
else:
    print(f"\nReading images from '{drive_folder_path}'.")
    # Collect every image file in the folder
    image_files = [os.path.join(drive_folder_path, f) for f in os.listdir(drive_folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
    image_files = natsorted(image_files) # natural sort of image paths

    if not image_files:
        print("No image files found in the folder. Make sure it contains images.")
    else:
        print(f"Found {len(image_files)} images; processing in order...")

        # Holds ground truth, OCR text, prediction and correctness per image
        evaluation_results = []
        total_processed_images = 0 # images actually processed (valid labels only)
        successful_ocr_count = 0
        successful_api_calls = 0

        # Iterate over every image in the folder
        for i, full_image_path in enumerate(image_files):
            image_file_name = os.path.basename(full_image_path)
            print(f"\n--- Processing image {i+1}/{len(image_files)}: '{image_file_name}' ---")

            # Derive the ground-truth label from the filename
            true_label_full = "未知對話" # full ground-truth label
            true_label_simplified = "未知" # simplified ground-truth label

            # Filenames containing the normal marker are legitimate; those with the scam marker are fraudulent.
            if re.search(r'正常對話', image_file_name, re.IGNORECASE):
                true_label_full = "正常對話"
                true_label_simplified = "正常"
            elif re.search(r'詐騙對話', image_file_name, re.IGNORECASE):
                true_label_full = "詐騙對話"
                true_label_simplified = "詐騙"
            else:
                print(f"Warning: filename '{image_file_name}' could not be resolved to a ground-truth label; excluded from the accuracy calculation.")
                continue # skip images without a resolvable label

            total_processed_images += 1 # only count images with a resolvable label

            # --- 4. Run OCR to extract text from the image ---
            print("Running OCR, please wait...")
            ocr_text = "OCR 失敗或無文字" # initialise to the error state
            try:
                img = Image.open(full_image_path)
                # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
                ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
                ocr_text = ocr_text.strip() # strip surrounding whitespace

                # Check whether the OCR output is long enough to be usable
                if ocr_text and len(ocr_text) > 10: # minimum length threshold to reject near-empty OCR output
                    print("OCR succeeded.")
                    successful_ocr_count += 1
                else:
                    print(f"OCR ran, but the extracted text was too short or empty: '{ocr_text}'。")
                    ocr_text = "未辨識出有效文字" # clearer sentinel value

            except Exception as e:
                print(f"OCR failed for image '{image_file_name}'：{e}")
                ocr_text = f"OCR 錯誤: {e}" # include the error message

            print(f"OCR output (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")


            # --- 5. Send the OCR output to the fine-tuned model ---
            model_prediction_raw = "模型未回應" # raw model response
            model_prediction_simplified = "未知" # simplified model prediction
            is_correct = False

            # Only call the model when OCR produced usable text
            if ocr_text and ocr_text != "未辨識出有效文字" and not ocr_text.startswith("OCR 錯誤"):
                print("Sending text to the fine-tuned model for classification, please wait...")
                time.sleep(1) # brief pause for readability

                try:
                    response_model = client.chat.completions.create(
                        model=fine_tuned_model_id,
                        messages=[
                            {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一段聊天對話文字，並將其判斷為『詐騙對話』還是『正常對話』。請只輸出『詐騙對話』或『正常對話』，不要包含任何額外文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                            {"role": "user", "content": "請根據以下聊天對話內容判斷是詐騙對話還是正常對話：" + ocr_text} # use the OCR text with a refined user prompt
                        ],
                        max_tokens=50, # cap the output length
                        temperature=0 # temperature 0 for deterministic output
                    )

                    model_prediction_raw = response_model.choices[0].message.content.strip() # read the response and strip whitespace
                    model_prediction_simplified = get_simplified_label(model_prediction_raw) # simplify the prediction
                    successful_api_calls += 1

                    # --- 6. Display the final classification ---
                    print("\n--- Final classification ---")
                    # Compare the simplified ground truth with the simplified prediction
                    if true_label_simplified == model_prediction_simplified and true_label_simplified != "未知":
                        is_correct = True
                    else:
                        is_correct = False

                    if "詐騙" in model_prediction_simplified: # check whether the simplified label indicates a scam
                        print(f"High probability of a scam. Proceed with caution. (prediction: '{model_prediction_raw}')")
                    elif "正常" in model_prediction_simplified: # check whether the simplified label indicates a normal conversation
                        print(f"Low probability of a scam. Safe to continue. (prediction: '{model_prediction_raw}')")
                    else:
                        print(f"Model response was ambiguous: '{model_prediction_raw}'。Check the model output.")

                except Exception as e:
                    print(f"Error calling the fine-tuned model for image '{image_file_name}'：{e}")
                    print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")
                    model_prediction_raw = f"API 錯誤: {e}" # mark the prediction as failed
                    model_prediction_simplified = "API 錯誤" # mark the simplified label as failed

            else:
                model_prediction_raw = "OCR 無法提供有效文字，未呼叫模型"
                model_prediction_simplified = "無文字" # simplified label
                print("OCR produced no usable text; skipping the model call.")

            # Append this result to the list
            evaluation_results.append({
                '檔名': image_file_name,
                '真實標籤': true_label_full, # 儲存原始完整標籤
                'OCR內容': ocr_text[:200], # first 200 chars
                '模型預測': model_prediction_raw, # raw model response
                '是否準確': '是' if is_correct else '否'
            })

            print("\n" + "="*50 + "\n") # separator between images

        print("\nAll images processed.")

        # --- 5. Display results and compute overall accuracy ---
        if not evaluation_results:
            print("No results to score. Check that the folder contains images with correctly formatted filenames.")
        else:
            correct_count = sum(1 for res in evaluation_results if res['是否準確'] == '是')

            # Count images eligible for scoring
            # Validity is based on the simplified prediction
            effective_evaluations = sum(1 for res in evaluation_results if get_simplified_label(res['模型預測']) in ["詐騙", "正常"])

            accuracy = (correct_count / effective_evaluations) * 100 if effective_evaluations > 0 else 0

            print("\n--- Detailed evaluation results ---")
            # Convert results to a DataFrame for display
            results_df = pd.DataFrame(evaluation_results)
            from IPython.display import display
            display(results_df)

            print("\n--- Overall accuracy ---")
            print(f"Images processed (with valid labels): {total_processed_images}")
            print(f"Images with successful OCR: {successful_ocr_count}")
            print(f"Successful model API calls: {successful_api_calls}")
            print(f"Images counted toward accuracy (model returned a clear prediction): {effective_evaluations}")
            print(f"Correctly classified images: {correct_count}")
            print(f"Final accuracy: {accuracy:.2f}%")

In [ ]:
import pandas as pd
import json
import os

# Assumes the CSV has been loaded into a DataFrame named 'df'
# If the session was restarted, re-run the CSV loading cell:
# from google.colab import files
# print("Upload your CSV file.")
# uploaded = files.upload()
# csv_file_name = next(iter(uploaded)) # get the uploaded filename
# df = pd.read_csv(csv_file_name, encoding='Big5') # use the correct encoding

# --- Inspect DataFrame column names (optional, for debugging) ---
print("DataFrame columns:")
print(df.columns)
print("\n--- Formatting data ---")

# Create a list to hold each formatted dialogue
my_jsonl_data = []

for index, row in df.iterrows():
    dialogue_entry = {
        "messages": [
            {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。你可以「特別注意連結，注意陌生訊息，或是異常或非常規的金錢要求。"},
            {"role": "user", "content":"對話內容是：" + str(row['對話內容'])}, # <--- maps to the conversation-text column
            {"role": "assistant", "content": str(row['對話類型'])} # <--- maps to the label column
        ]
    }
    my_jsonl_data.append(dialogue_entry)

# Print the first record to verify the format
if my_jsonl_data:
    print("\nFirst formatted record:")
    print(my_jsonl_data[0])
else:
    print("\nNothing was formatted. Check whether the DataFrame is empty.")


# Save as a JSONL file:
output_jsonl_path = '/content/fine_tuning_data.jsonl'
try:
    with open(output_jsonl_path, 'w', encoding='utf-8') as f:
        for entry in my_jsonl_data:
            json.dump(entry, f, ensure_ascii=False) # ensure_ascii=False keeps Chinese characters readable
            f.write('\n') # one JSON object per line

    print(f"\nData saved to '{output_jsonl_path}'")
    print("You can download it from the Files panel on the left in Colab.")
except Exception as e:
    print(f"\nError saving the JSONL file: {e}")

In [ ]:
import pandas as pd
import json
import os

# --- Make sure the DataFrame 'df' is loaded ---
# If the session was restarted, re-run the CSV loading step
# from google.colab import files
# print("Upload your CSV file.")
# uploaded = files.upload()
# csv_file_name = next(iter(uploaded)) # get the uploaded filename
# df = pd.read_csv(csv_file_name, encoding='Big5') # use the correct encoding

# Assumes 'df' holds the conversation text and label columns

# --- Format the data as JSONL ---
my_jsonl_data = []

for index, row in df.iterrows():
    dialogue_entry = {
        "messages": [
            {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"},
            {"role": "user", "content": "對話內容是：" + str(row['對話內容'])},
            {"role": "assistant", "content": str(row['對話類型'])}
        ]
    }
    my_jsonl_data.append(dialogue_entry)

# --- Preview the JSONL content in Colab ---
print("\n--- Formatted JSONL preview (first 5 records) ---")
# Print only the first few records to keep the output short
for i, entry in enumerate(my_jsonl_data):
    if i >= 5: # show only the first 5
        break
    print(json.dumps(entry, indent=2, ensure_ascii=False)) # indent=2 improves readability，ensure_ascii=False keeps Chinese characters readable
    print("---") # separator between records

if len(my_jsonl_data) > 5:
    print(f"\n... {len(my_jsonl_data) - 5} more records not shown.")

# To print every record (may be very long)
# print("\n--- All formatted JSONL records ---")
# for entry in my_jsonl_data:
#     print(json.dumps(entry, ensure_ascii=False))

In [ ]:
client.files.create(
  file=open("/content/fine_tuning_data.jsonl", "rb"),
  purpose="fine-tune"
)

In [ ]:
client.fine_tuning.jobs.create(
  training_file="file-JwViajXesxmq6G69Fywurt",
  model="gpt-3.5-turbo-1106"
)

In [ ]:
import os
import time
from openai import OpenAI

# Make sure your API key is set
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # use your own API key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


# Fine-tuning job ID from the previous cell output
fine_tuning_job_id = 'ftjob-sV3YmTDjbJDuxlL7Nx8rStfP'

print(f"Checking fine-tuning job '{fine_tuning_job_id}' status...")

# Poll until the job succeeds or fails
while True:
    job = client.fine_tuning.jobs.retrieve(fine_tuning_job_id)
    print(f"Current status: {job.status}")

    if job.status == 'succeeded':
        print("\n--- Fine-tuning job completed successfully. ---")
        print(f"Fine-tuned model ID: {job.fine_tuned_model}")
        # use job.fine_tuned_model to call the fine-tuned model
        break
    elif job.status == 'failed':
        print("\n--- Fine-tuning job failed. ---")
        if job.error:
            print(f"Error message: {job.error.message}")
        break
    elif job.status in ['validating_files', 'queued', 'running']:
        print("Job still running; checking again in 30 seconds...")
        time.sleep(30) # wait before checking again
    else:
        print(f"Unknown status: {job.status}. Checking again in 30 seconds...")
        time.sleep(30)

# 詐騙與正常訊息辨識(嘗試) gpt-3.5-turbo

In [ ]:
import os
from openai import OpenAI

# Make sure your API key is set
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # replace with your own API key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # use your own API key

# Fine-tuned model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- Example 1: Test a normal conversation ---
normal_conversation_text = "不好意思，商品售罄"

print(f"Test conversation: '{normal_conversation_text}'")

try:
    response_normal = client.chat.completions.create(
        model=fine_tuned_model_id, # <--- the fine-tuned model ID
        messages=[
            # system and user prompts must match the fine-tuning format
            {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。你可以「特別注意連結，注意陌生訊息，或是異常或非常規的金錢要求。"},
            {"role": "user", "content": "對話內容是：" + normal_conversation_text}
            # a fine-tuned model needs no assistant example; it generates directly
        ],
        max_tokens=50 # cap the output length; only a single label is expected
    )

    prediction_normal = response_normal.choices[0].message.content
    print(f"Model prediction: {prediction_normal}")

except Exception as e:
    print(f"Error calling the model: {e}")
    print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to use it.")


print("\n" + "="*50 + "\n")

# --- Example 2: Test a likely scam conversation ---
scam_conversation_text = "我需要你點擊此連結：http://bit.ly/scam-link來付款喔"

print(f"Test conversation: '{scam_conversation_text}'")

try:
    response_scam = client.chat.completions.create(
        model=fine_tuned_model_id, # <--- the fine-tuned model ID
        messages=[
            {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。你可以「特別注意連結，注意陌生訊息，或是異常或非常規的金錢要求。"},
            {"role": "user", "content": "對話內容是：" + scam_conversation_text}
        ],
        max_tokens=50
    )

    prediction_scam = response_scam.choices[0].message.content
    print(f"Model prediction: {prediction_scam}")

except Exception as e:
    print(f"Error calling the model: {e}")
    print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to use it.")

In [ ]:
import os
from openai import OpenAI

# Make sure your API key is set
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # replace with your own API key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # use your own API key

# Fine-tuned model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- Test a normal conversation ---
normal_conversation_text = "不好意思，您要的商品已經售罄了"

print(f"Test conversation: '{normal_conversation_text}'")

try:
    response_normal = client.chat.completions.create(
        model=fine_tuned_model_id, # <--- the fine-tuned model ID
        messages=[
            # system and user prompts must match the fine-tuning format
            {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。你可以「特別注意連結，注意陌生訊息，或是異常或非常規的金錢要求。"},
            {"role": "user", "content": "對話內容是：" + normal_conversation_text}
            # a fine-tuned model needs no assistant example; it generates directly
        ],
        max_tokens=50 # cap the output length; only a single label is expected
    )

    prediction_normal = response_normal.choices[0].message.content
    print(f"Model prediction: {prediction_normal}")

except Exception as e:
    print(f"Error calling the model: {e}")
    print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to use it.")


print("\n" + "="*50 + "\n")

# --- Test a likely scam conversation ---
scam_conversation_text = "我需要你點擊此連結：http://bit.ly/scam-link來進行付款喔"

print(f"Test conversation: '{scam_conversation_text}'")

try:
    response_scam = client.chat.completions.create(
        model=fine_tuned_model_id, # <--- the fine-tuned model ID
        messages=[
            {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。你可以「特別注意連結，注意陌生訊息，或是異常或非常規的金錢要求。"},
            {"role": "user", "content": "對話內容是：" + scam_conversation_text}
        ],
        max_tokens=50
    )

    prediction_scam = response_scam.choices[0].message.content
    print(f"Model prediction: {prediction_scam}")

except Exception as e:
    print(f"Error calling the model: {e}")
    print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to use it.")

In [ ]:
import os
from openai import OpenAI

# Make sure your API key is set
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # replace with your own API key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # use your own API key

# Fine-tuned model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- Test a normal conversation ---
normal_conversation_text = "不好意思，您要的商品已經售完了"

print(f"Test conversation: '{normal_conversation_text}'")

try:
    response_normal = client.chat.completions.create(
        model=fine_tuned_model_id, # <--- the fine-tuned model ID
        messages=[
            # system and user prompts must match the fine-tuning format
            {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。你可以「特別注意連結，注意陌生訊息，或是異常或非常規的金錢要求。"},
            {"role": "user", "content": "對話內容是：" + normal_conversation_text}
            # a fine-tuned model needs no assistant example; it generates directly
        ],
        max_tokens=50 # cap the output length; only a single label is expected
    )

    prediction_normal = response_normal.choices[0].message.content
    print(f"Model prediction: {prediction_normal}")

except Exception as e:
    print(f"Error calling the model: {e}")
    print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to use it.")


print("\n" + "="*50 + "\n")

# --- Test a likely scam conversation ---
scam_conversation_text = "您好，請您點擊連結添加平台委託的金融機構簽署專員的賴來協助您完成認證簽署，預計10分鐘內操作完成認證。連結如下：https://line.me/ti/p/9EgWEJEkbl"

print(f"Test conversation: '{scam_conversation_text}'")

try:
    response_scam = client.chat.completions.create(
        model=fine_tuned_model_id, # <--- the fine-tuned model ID
        messages=[
            {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。你可以「特別注意連結，注意陌生訊息，或是異常或非常規的金錢要求。"},
            {"role": "user", "content": "對話內容是：" + scam_conversation_text}
        ],
        max_tokens=50
    )

    prediction_scam = response_scam.choices[0].message.content
    print(f"Model prediction: {prediction_scam}")

except Exception as e:
    print(f"Error calling the model: {e}")
    print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to use it.")

In [ ]:
import os
from openai import OpenAI

# Make sure your API key is set
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # replace with your own API key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # use your own API key

# Fine-tuned model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- Test a normal conversation ---
normal_conversation_text = "不好意思，您要的商品已經沒了"

print(f"Test conversation: '{normal_conversation_text}'")

try:
    response_normal = client.chat.completions.create(
        model=fine_tuned_model_id, # <--- the fine-tuned model ID
        messages=[
            # system and user prompts must match the fine-tuning format
            {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。你可以「特別注意連結，注意陌生訊息，或是異常或非常規的金錢要求。"},
            {"role": "user", "content": "對話內容是：" + normal_conversation_text}
            # a fine-tuned model needs no assistant example; it generates directly
        ],
        max_tokens=50 # cap the output length; only a single label is expected
    )

    prediction_normal = response_normal.choices[0].message.content
    print(f"Model prediction: {prediction_normal}")

except Exception as e:
    print(f"Error calling the model: {e}")
    print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to use it.")


print("\n" + "="*50 + "\n")

# --- Test a likely scam conversation ---
scam_conversation_text = "https://m.ecpaygtw.com/s/oy1UyI47。麻煩您使用此連結來填入收款交易資訊"

print(f"Test conversation: '{scam_conversation_text}'")

try:
    response_scam = client.chat.completions.create(
        model=fine_tuned_model_id, # <--- the fine-tuned model ID
        messages=[
            {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。你可以「特別注意連結，注意陌生訊息，或是異常或非常規的金錢要求。"},
            {"role": "user", "content": "對話內容是：" + scam_conversation_text}
        ],
        max_tokens=50
    )

    prediction_scam = response_scam.choices[0].message.content
    print(f"Model prediction: {prediction_scam}")

except Exception as e:
    print(f"Error calling the model: {e}")
    print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to use it.")

# 用戶端：上傳可疑對話截圖->OCR->model process->result (with gpt-3.5-turbo)

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

In [ ]:
from openai import OpenAI
import os
import time # for pacing the output

# --- 1. Configure the API key and fine-tuned model ID ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-Tune model ID
fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF"

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Let the user upload an image ---
print("\nUpload the chat screenshot you want to classify.")
uploaded_files = files.upload() # opens a file picker

if not uploaded_files:
    print("No file uploaded. Re-run the cell and upload an image.")
else:
    # take the first uploaded file
    image_file_name = next(iter(uploaded_files))
    print(f"File '{image_file_name}'  uploaded successfully.")

    # Save the uploaded file into the Colab workspace
    uploaded_image_path = os.path.join("/content/", image_file_name)
    with open(uploaded_image_path, 'wb') as f:
        f.write(uploaded_files[image_file_name])

    # --- 4. Run OCR to extract text from the image ---
    print("\nRunning OCR, please wait...")
    try:
        img = Image.open(uploaded_image_path)
        # Run Tesseract OCR with the Traditional Chinese pack ('chi_tra')
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip() # strip surrounding whitespace

        if ocr_text:
            print("OCR succeeded. Extracted text (first 200 chars):")
            print("---")
            print(ocr_text[:200] + ("..." if len(ocr_text) > 200 else ""))
            print("---")
        else:
            print("OCR ran, but no text was detected. The image text may be unclear.")
            ocr_text = "無法辨識文字內容" # default value so the model never receives empty input

    except Exception as e:
        print(f"OCR failed: {e}")
        ocr_text = "OCR 錯誤：無法辨識文字內容" # error handling

    # --- 5. Send the OCR output to the fine-tuned model ---
    print("\nSending text to the fine-tuned model for classification, please wait...")
    time.sleep(2) # brief pause for readability

    try:
        response_model = client.chat.completions.create(
            model=fine_tuned_model_id,
            messages=[
                {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"}, # keep the system prompt identical to the one used during fine-tuning
                {"role": "user", "content": "對話內容是：" + ocr_text} # use the OCR text
            ],
            max_tokens=50 # cap the output length
        )

        model_prediction = response_model.choices[0].message.content.strip() # read the response and strip whitespace

        # --- 6. Display the final classification ---
        print("\n--- Final classification ---")
        if "詐騙" in model_prediction: # check whether the response indicates a scam
            print("High probability of a scam. Proceed with caution.")
        elif "正常" in model_prediction: # check whether the response indicates a normal conversation
            print("Low probability of a scam. Safe to continue.")
        else:
            print(f"Model response was ambiguous: '{model_prediction}'。Check the model output.")

    except Exception as e:
        print(f"Error calling the fine-tuned model: {e}")
        print("Check that your API key is valid, the fine-tuned model ID is correct, and you have permission to call it.")

# gpt-3.5-turbo (106測試資料)

In [ ]:
import os
import re
from PIL import Image
import pytesseract
from google.colab import files # kept in case manual upload is needed
from openai import OpenAI
import time
from natsort import natsorted # install if missing: !pip install natsort
import pandas as pd # pandas, for displaying results

# --- 1. Configure the API key and fine-tuned model ID ---
# Make sure your API key is set and has permission to call the model
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Using base gpt-3.5-turbo; swap in the fine-tuned model ID if needed
fine_tuned_model_id = "gpt-3.5-turbo" # base gpt-3.5-turbo

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Mount Google Drive and copy the dataset folder ---
# Make sure Google Drive is mounted
from google.colab import drive
drive.mount('/content/drive')

# Path to the dataset folder in Google Drive (adjust to your own folder structure)
drive_folder_path = '/content/drive/MyDrive/大三/下/AI導論/test images for 3o and 4.1'
# Target path inside Colab (copied under /content/, keeping the folder name)
colab_base_path = '/content/'
colab_folder_name = os.path.basename(drive_folder_path)
colab_full_path = os.path.join(colab_base_path, colab_folder_name)

# Create the target folder in Colab if it does not exist
os.makedirs(colab_full_path, exist_ok=True)

# Copy the files
print(f"Copying from '{drive_folder_path}' to '{colab_full_path}'...")
# cp -r copies the folder and its contents
!cp -r "{drive_folder_path}/." "{colab_full_path}/" # copy folder contents to the target directory
print("Folder copy complete.")

# --- 4. Batch-process images, classify, and score accuracy ---
# Collect paths to every image in the folder
image_files = [os.path.join(colab_full_path, f) for f in os.listdir(colab_full_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
image_files = natsorted(image_files) # natural sort for consistent ordering

print(f"\n'{colab_full_path}' contains {len(image_files)} images.")
if not image_files:
    print("Error: no image files found. Check the folder path and file extensions.")
    print("Supported formats are .png, .jpg, .jpeg, .gif and .bmp.")

# Holds ground truth, prediction and correctness per image
evaluation_results = []
total_processed = 0
successful_ocr_count = 0 # OCR counter
successful_api_calls = 0

print("\nStarting batch processing. This may take a while...")

for i, image_path in enumerate(image_files):
    filename = os.path.basename(image_path)
    print(f"\n--- Processing image {i+1}/{len(image_files)}: {filename} ---")

    # Derive the ground-truth label from the filename
    true_label = "未知"
    if "normal" in filename.lower():
        true_label = "正常對話"
    elif "scam_test" in filename.lower():
        true_label = "詐騙對話"
    else:
        print(f"Warning: filename '{filename}' could not be resolved to a ground-truth label; skipping.")
        continue

    total_processed += 1

    # --- Re-introduce the OCR step ---
    ocr_text = "OCR 失敗"
    try:
        img = Image.open(image_path)
        ocr_text = pytesseract.image_to_string(img, lang='chi_tra')
        ocr_text = ocr_text.strip()
        if ocr_text:
            print("  OCR succeeded.")
            successful_ocr_count += 1
        else:
            print("  OCR ran, but no text was detected.")
            ocr_text = "未辨識出文字" # default value
    except Exception as e:
        print(f"  OCR failed: {e}")

    print(f"  OCR output (first 100 chars): '{ocr_text[:100]}'...")

    # --- Send the OCR output to gpt-3.5-turbo ---
    model_prediction = "模型未回應"
    is_correct = False

    # Only call the model when OCR produced some text
    if ocr_text and ocr_text != "未辨識出文字" and ocr_text != "OCR 失敗":
        try:
            response_model = client.chat.completions.create(
                model=fine_tuned_model_id, # using gpt-3.5-turbo
                messages=[
                    {"role": "system", "content": "你是一個詐騙與正常訊息辨識助手。你會收到一條對話訊息，並將其判斷為詐騙對話還是正常對話。請只回覆 '詐騙對話' 或 '正常對話'，不要包含其他任何資訊、原始文字或解釋。"},
                    {"role": "user", "content": "請辨識這段文字內容是詐騙對話還是正常對話：" + ocr_text}
                ],
                max_tokens=50, # cap the output length
            )
            model_prediction = response_model.choices[0].message.content.strip()
            successful_api_calls += 1
            print(f"  Raw model prediction: '{model_prediction}'")

            # Correctness check
            if "正常" in true_label and "正常" in model_prediction:
                is_correct = True
            elif "詐騙" in true_label and "詐騙" in model_prediction:
                is_correct = True

        except Exception as e:
            print(f"  Model call failed: {e}")
            model_prediction = f"API 呼叫失敗: {e}"
    else:
        model_prediction = "OCR 無法辨識，未呼叫模型"
        print("  OCR failed; skipping the model call.")


    evaluation_results.append({
        '檔名': filename,
        '真實標籤': true_label,
        'OCR內容': ocr_text[:100], # first 100 chars only
        '模型預測': model_prediction,
        '是否準確': '是' if is_correct else '否'
    })

    # brief pause to stay within API rate limits
    time.sleep(0.5)

print("\nAll images processed.")

# --- 5. Display results and compute overall accuracy ---
if not evaluation_results:
    print("No results to score. Check that the folder contains images with correctly formatted filenames.")
else:
    correct_count = sum(1 for res in evaluation_results if res['是否準確'] == '是')
    total_evaluations = len(evaluation_results) # number of images with valid labels

    # Only score images where the model returned a clear prediction
    effective_evaluations = sum(1 for res in evaluation_results if res['模型預測'] not in ["模型未回應", "API 呼叫失敗", "OCR 無法辨識，未呼叫模型"])

    accuracy = (correct_count / effective_evaluations) * 100 if effective_evaluations > 0 else 0

    print("\n--- Detailed evaluation results ---")
    results_df = pd.DataFrame(evaluation_results)
    from IPython.display import display
    display(results_df)

    print("\n--- Overall accuracy ---")
    print(f"Images processed (valid labels): {total_processed}")
    print(f"Images with successful OCR: {successful_ocr_count}")
    print(f"Successful model API calls: {successful_api_calls}")
    print(f"Images counted toward accuracy (model call succeeded): {effective_evaluations}")
    print(f"Correctly classified images: {correct_count}")
    print(f"Final accuracy: {accuracy:.2f}%")

# gpt4o model (104初始資料)

In [ ]:
import base64
import os
from openai import OpenAI
from PIL import Image # For image processing (resizing, converting to Base64)
import io # To handle image in memory

# Initialize your client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Function to encode image to base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Your local folder where images are stored (from previous steps)
image_folder_path = '/content/詐騙正常對話資料集_colab' # Make sure this path is correct

# Get list of image files
image_files = [os.path.join(image_folder_path, f) for f in os.listdir(image_folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
from natsort import natsorted
image_files = natsorted(image_files) # Ensure consistent order

result = []

print("Classifying images with the multimodal model...")

# Limit for demonstration, remove for all 107 images
# For testing, you might want to process a smaller subset first
# image_files = image_files[:5]

for i, image_path in enumerate(image_files):
    print(f"Processing image {i+1}/{len(image_files)}: {os.path.basename(image_path)}")
    try:
        # Encode the image
        base64_image = encode_image(image_path)

        # Make the API call with the image data
        response = client.chat.completions.create(
            # Use a multimodal model
            model="gpt-4o", # Recommended for its vision capabilities and cost-effectiveness
            # Or model="gpt-4-turbo-2024-04-09", # Another strong option for vision
            messages=[
                {
                    "role": "system",
                    "content": "你是一個人工智慧詐騙與正常訊息辨識助手。請你辨識我提供的圖片內容是詐騙對話還是正常對話。請你不要加入其他訊息與資訊，只要給我最原始的資料即可，不用做過多解釋。例如：'這是一段詐騙對話' 或 '這是一段正常對話'。",
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "請辨識這張圖片中的對話內容是詐騙還是正常:"},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}",
                                "detail": "high" # Use "high" for better detail, but costs more. "low" is faster/cheaper.
                            },
                        },
                    ],
                },
            ],
            max_tokens=100, # Limit the response length
        )

        content = response.choices[0].message.content
        print(f"Prediction: {content}")
        result.append(content)

    except Exception as e:
        error_message = f"處理圖片 {os.path.basename(image_path)} 時發生錯誤: {e}"
        print(error_message)
        result.append(error_message)

print("\nAll images processed.")
print("Results:")
for res in result:
    print(res)

In [ ]:
import os
import re # regular expressions

# Assumes 'image_files' and 'result' were populated by the cells above.
# image_files holds full image paths, e.g.
# image_files = [
#     '/content/詐騙正常對話資料集_colab/正常對話1.jpg',
#     '/content/詐騙正常對話資料集_colab/正常對話2.jpg',
#     # ... other image paths
# ]

# result holds the model's output for each image, e.g.
# result = [
#     'this is a scam conversation',
#     'this is a normal conversation',
#     # ... other results
# ]

# Check that result and image_files have the same length
if len(image_files) != len(result):
    print("Error: the image list and the results list have different lengths; accuracy cannot be computed.")
else:
    total_images = len(image_files)
    correct_predictions = 0

    print("\n--- Calculating accuracy ---")
    print("-------------------------------------------------------------------------------------------------------")
    print(f"{'filename':<30} | {'ground truth':<10} | {'prediction':<15} | {'correct':<10}")
    print("-------------------------------------------------------------------------------------------------------")

    for i in range(total_images):
        file_path = image_files[i]
        model_output = result[i]

        # Derive the ground-truth label from the filename
        filename = os.path.basename(file_path)
        true_label = "未知" # default

        # Resolve the ground truth with a regex or simple string check
        # match the normal or scam marker, ignoring case and digits
        if re.search(r'正常對話', filename, re.IGNORECASE):
            true_label = "正常對話"
        elif re.search(r'詐騙對話', filename, re.IGNORECASE):
            true_label = "詐騙對話"

        # Extract the predicted label from the model output
        predicted_label = "未知"
        if "正常對話" in model_output:
            predicted_label = "正常對話"
        elif "詐騙對話" in model_output:
            predicted_label = "詐騙對話"

        # Correctness check
        is_correct = False
        if true_label == "正常對話" and predicted_label == "正常對話":
            is_correct = True
        elif true_label == "詐騙對話" and predicted_label == "詐騙對話":
            is_correct = True

        if is_correct:
            correct_predictions += 1

        print(f"{filename:<30} | {true_label:<10} | {predicted_label:<15} | {'是' if is_correct else '否':<10}")

    print("-------------------------------------------------------------------------------------------------------")

    # Compute accuracy
    accuracy = (correct_predictions / total_images) * 100 if total_images > 0 else 0

    print(f"\nTotal images: {total_images}")
    print(f"Correct predictions: {correct_predictions}")
    print(f"Accuracy: {accuracy:.2f}%")

# gpt4o model（18測試資料）

In [ ]:
import os
import re
from PIL import Image
import pytesseract
from google.colab import files
from openai import OpenAI
import time
from natsort import natsorted # install if missing: !pip install natsort
import pandas as pd # pandas, for displaying results

# --- 1. Configure the API key and fine-tuned model ID ---
# Make sure your API key is set and has permission to call the model
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-tuned model ID (not used here; gpt-4o performs the visual classification)
# fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF" # unused here; gpt-4o handles the images directly

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Mount Google Drive and copy the dataset folder ---
# Make sure Google Drive is mounted
from google.colab import drive
drive.mount('/content/drive')

# Path to the dataset folder in Google Drive (adjust to your own folder structure)
drive_folder_path = '/content/drive/MyDrive/大三/下/AI導論/test images for 3o and 4.1'
# Target path inside Colab (copied under /content/, keeping the folder name)
colab_base_path = '/content/'
colab_folder_name = os.path.basename(drive_folder_path)
colab_full_path = os.path.join(colab_base_path, colab_folder_name)

# Create the target folder in Colab if it does not exist
os.makedirs(colab_full_path, exist_ok=True)

# Copy the files
print(f"Copying from '{drive_folder_path}' to '{colab_full_path}'...")
# cp -r copies the folder and its contents
!cp -r "{drive_folder_path}/." "{colab_full_path}/" # copy folder contents to the target directory
print("Folder copy complete.")

# --- 4. Batch-process images, classify, and score accuracy ---
# Collect paths to every image in the folder
image_files = [os.path.join(colab_full_path, f) for f in os.listdir(colab_full_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
image_files = natsorted(image_files) # natural sort for consistent ordering

print(f"\n'{colab_full_path}' contains {len(image_files)} images.")
if not image_files:
    print("Error: no image files found. Check the folder path and file extensions.")
    print("Supported formats are .png, .jpg, .jpeg, .gif and .bmp.")

# Holds ground truth, prediction and correctness per image
evaluation_results = []
total_processed = 0
# successful_ocr_count = 0 # no OCR counter needed; the vision model reads the image directly
successful_api_calls = 0

print("\nStarting batch visual classification. This may take a while...")

# base64 helpers
import base64
import io

# Helper: encode the image as base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

for i, image_path in enumerate(image_files):
    filename = os.path.basename(image_path)
    print(f"\n--- Processing image {i+1}/{len(image_files)}: {filename} ---")

    # Derive the ground-truth label from the filename
    true_label = "未知"
    # Ground truth is parsed from the filename.
    # Filenames containing 'normal' are legitimate; those containing 'scam_test' are fraudulent.
    if "normal" in filename.lower():
        true_label = "正常對話"
    elif "scam_test" in filename.lower():
        true_label = "詐騙對話"
    else:
        print(f"Warning: filename '{filename}' could not be resolved to a ground-truth label from the 'normal' or 'scam_test' keyword; skipping.")
        continue # skip images without a resolvable label

    total_processed += 1

    # Classify the image directly with the multimodal model
    model_prediction = "模型未回應"
    is_correct = False

    try:
        # encode the image as base64
        base64_image = encode_image(image_path)

        response_model = client.chat.completions.create(
            model="gpt-4o", # gpt-4o handles the image directly
            messages=[
                {
                    "role": "system",
                    "content": "你是一個人工智慧詐騙與正常訊息辨識助手。請你辨識我提供的圖片內容是詐騙對話還是正常對話。請你不要加入其他訊息與資訊，只要給我最原始的資料即可，不用做過多解釋。例如：'這是一段詐騙對話' 或 '這是一段正常對話'。", # matched to the gpt-4o prompt
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "請辨識這張圖片中的對話內容是詐騙還是正常:"},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}",
                                "detail": "high" # 'high' gives better fidelity at higher cost
                            },
                        },
                    ],
                },
            ],
            max_tokens=100, # cap the output length
        )
        model_prediction = response_model.choices[0].message.content.strip()
        successful_api_calls += 1
        print(f"  Raw model prediction: '{model_prediction}'")

        # --- More flexible correctness check ---
        # Correctness check
        is_correct = False
        # check whether ground truth and prediction agree on the label
        if "正常" in true_label and "正常" in model_prediction:
            is_correct = True
        elif "詐騙" in true_label and "詐騙" in model_prediction:
            is_correct = True
        # --- End of correctness check ---

    except Exception as e:
        print(f"  Model call failed: {e}")
        model_prediction = f"API 呼叫失敗或圖片處理錯誤: {e}"

    evaluation_results.append({
        '檔名': filename,
        '真實標籤': true_label,
        '模型預測': model_prediction,
        '是否準確': '是' if is_correct else '否'
    })

    # brief pause to stay within API rate limits
    time.sleep(0.5)

print("\nAll images processed.")

# --- 5. Display results and compute overall accuracy ---
if not evaluation_results:
    print("No results to score. Check that the folder contains images with correctly formatted filenames.")
else:
    correct_count = sum(1 for res in evaluation_results if res['是否準確'] == '是')
    total_evaluations = len(evaluation_results)

    accuracy = (correct_count / total_evaluations) * 100 if total_evaluations > 0 else 0

    print("\n--- Detailed evaluation results ---")
    # Convert results to a DataFrame for display
    results_df = pd.DataFrame(evaluation_results)
    from IPython.display import display
    display(results_df)

    print("\n--- Overall accuracy ---")
    print(f"Images processed (valid labels): {total_processed}")
    print(f"Successful model API calls: {successful_api_calls}")
    print(f"Correctly classified images: {correct_count}")
    print(f"Final accuracy: {accuracy:.2f}%")

# gpt4o model (106測試資料)

In [ ]:
# 106 test
import os
import re
from PIL import Image
import pytesseract
from google.colab import files
from openai import OpenAI
import time
from natsort import natsorted # install if missing: !pip install natsort
import pandas as pd # pandas, for displaying results

# --- 1. Configure the API key and fine-tuned model ID ---
# Make sure your API key is set and has permission to call the model
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Fine-tuned model ID (not used here; gpt-4o performs the visual classification)
# fine_tuned_model_id = "ft:gpt-3.5-turbo-1106:ici::BdSO0syF" # unused here; gpt-4o handles the images directly

# --- 2. Ensure Tesseract OCR and the Traditional Chinese pack are installed ---
# Required for OCR
print("Checking and installing Tesseract OCR and the Traditional Chinese language pack...")
!apt update > /dev/null
!apt install tesseract-ocr -y > /dev/null
!apt install -y tesseract-ocr-chi-tra > /dev/null
!pip install pytesseract > /dev/null
print("OCR environment ready.")

# --- 3. Mount Google Drive and copy the dataset folder ---
# Make sure Google Drive is mounted
from google.colab import drive
drive.mount('/content/drive')

# Path to the dataset folder in Google Drive (adjust to your own folder structure)
drive_folder_path = '/content/drive/MyDrive/大三/下/AI導論/test images for 3o and 4.1'
# Target path inside Colab (copied under /content/, keeping the folder name)
colab_base_path = '/content/'
colab_folder_name = os.path.basename(drive_folder_path)
colab_full_path = os.path.join(colab_base_path, colab_folder_name)

# Create the target folder in Colab if it does not exist
os.makedirs(colab_full_path, exist_ok=True)

# Copy the files
print(f"Copying from '{drive_folder_path}' to '{colab_full_path}'...")
# cp -r copies the folder and its contents
!cp -r "{drive_folder_path}/." "{colab_full_path}/" # copy folder contents to the target directory
print("Folder copy complete.")

# --- 4. Batch-process images, classify, and score accuracy ---
# Collect paths to every image in the folder
image_files = [os.path.join(colab_full_path, f) for f in os.listdir(colab_full_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
image_files = natsorted(image_files) # natural sort for consistent ordering

print(f"\n'{colab_full_path}' contains {len(image_files)} images.")
if not image_files:
    print("Error: no image files found. Check the folder path and file extensions.")
    print("Supported formats are .png, .jpg, .jpeg, .gif and .bmp.")

# Holds ground truth, prediction and correctness per image
evaluation_results = []
total_processed = 0
# successful_ocr_count = 0 # no OCR counter needed; the vision model reads the image directly
successful_api_calls = 0

print("\nStarting batch visual classification. This may take a while...")

# base64 helpers
import base64
import io

# Helper: encode the image as base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

for i, image_path in enumerate(image_files):
    filename = os.path.basename(image_path)
    print(f"\n--- Processing image {i+1}/{len(image_files)}: {filename} ---")

    # Derive the ground-truth label from the filename
    true_label = "未知"
    # Ground truth is parsed from the filename.
    # Filenames containing 'normal' are legitimate; those containing 'scam_test' are fraudulent.
    if "normal" in filename.lower():
        true_label = "正常對話"
    elif "scam_test" in filename.lower():
        true_label = "詐騙對話"
    else:
        print(f"Warning: filename '{filename}' could not be resolved to a ground-truth label from the 'normal' or 'scam_test' keyword; skipping.")
        continue # skip images without a resolvable label

    total_processed += 1

    # Classify the image directly with the multimodal model
    model_prediction = "模型未回應"
    is_correct = False

    try:
        # encode the image as base64
        base64_image = encode_image(image_path)

        response_model = client.chat.completions.create(
            model="gpt-4o", # gpt-4o handles the image directly
            messages=[
                {
                    "role": "system",
                    "content": "你是一個人工智慧詐騙與正常訊息辨識助手。請你辨識我提供的圖片內容是詐騙對話還是正常對話。請你不要加入其他訊息與資訊，只要給我最原始的資料即可，不用做過多解釋。例如：'這是一段詐騙對話' 或 '這是一段正常對話'。", # matched to the gpt-4o prompt
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "請辨識這張圖片中的對話內容是詐騙還是正常:"},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}",
                                "detail": "high" # 'high' gives better fidelity at higher cost
                            },
                        },
                    ],
                },
            ],
            max_tokens=100, # cap the output length
        )
        model_prediction = response_model.choices[0].message.content.strip()
        successful_api_calls += 1
        print(f"  Raw model prediction: '{model_prediction}'")

        # --- More flexible correctness check ---
        # Correctness check
        is_correct = False
        # check whether ground truth and prediction agree on the label
        if "正常" in true_label and "正常" in model_prediction:
            is_correct = True
        elif "詐騙" in true_label and "詐騙" in model_prediction:
            is_correct = True
        # --- End of correctness check ---

    except Exception as e:
        print(f"  Model call failed: {e}")
        model_prediction = f"API 呼叫失敗或圖片處理錯誤: {e}"

    evaluation_results.append({
        '檔名': filename,
        '真實標籤': true_label,
        '模型預測': model_prediction,
        '是否準確': '是' if is_correct else '否'
    })

    # brief pause to stay within API rate limits
    time.sleep(0.5)

print("\nAll images processed.")

# --- 5. Display results and compute overall accuracy ---
if not evaluation_results:
    print("No results to score. Check that the folder contains images with correctly formatted filenames.")
else:
    correct_count = sum(1 for res in evaluation_results if res['是否準確'] == '是')
    total_evaluations = len(evaluation_results)

    accuracy = (correct_count / total_evaluations) * 100 if total_evaluations > 0 else 0

    print("\n--- Detailed evaluation results ---")
    # Convert results to a DataFrame for display
    results_df = pd.DataFrame(evaluation_results)
    from IPython.display import display
    display(results_df)

    print("\n--- Overall accuracy ---")
    print(f"Images processed (valid labels): {total_processed}")
    print(f"Successful model API calls: {successful_api_calls}")
    print(f"Correctly classified images: {correct_count}")
    print(f"Final accuracy: {accuracy:.2f}%")